# Project 5 — From Static to Contextual (UD English EWT): end-to-end Colab run
**Yellapu Dheeraj Babu (23CS10081)** · NLP CS60075 Assignment II

This notebook reproduces every result of the report:
* **Task 1:** PPMI + SVD vectors (WS-353, Google analogies, 20-sentence MST check).
* **Task 2:** the Hewitt & Manning structural probe on `bert-base-uncased` (layers 0–12 × ranks {32, 64, 128} × {distance, depth}).
* **Task 3:** BERT-derived static vectors from the best layer ℓ*, compared with Task 1.

**Before you start:** *Runtime → Change runtime type → GPU* (a T4 is enough).
Every step calls the same scripts as `python code/run_all.py`. All seeds are fixed (`SEED = 42` in `code/common/config.py`).

| Step | Approx. time on a T4 |
|---|---|
| Setup + data | 5–10 min |
| Task 1 | 1–2 min |
| Task 2: features | ~15 min |
| Task 2: 78 probes (`MODE="full"`) | ~4–6 h |
| Task 2: 26 probes (`MODE="fast"`, rank 64 only) | ~1.5–2 h |
| Task 3 | ~5 min |

Probe training is **resumable**: if the session disconnects, re-run the setup cells and the training cell, and finished probes are skipped.

## 0. Configuration

In [ ]:
MODE = "full"            # "full" = exactly the report (78 probes); "fast" = rank 64 only (26 probes)
WITH_CONTROL = False     # True = also train the spec-literal Task 2 control (constant lr, last epoch; +~1.5 h)
PIN_TORCH = True         # True = install the exact pinned torch from requirements.txt (reproducible, slower);
                         # False = keep Colab's preinstalled torch (faster; tiny numeric differences possible)
SAVE_TO_DRIVE = False    # True = copy outputs/ (without the 5 GB feature cache) to Google Drive at the end
PROJECT_DIR = "/content/project5"

## 1. Get the code
Upload the submission archive (`nlp_23cs10081_project5.zip`) when asked. Alternatively, put the unpacked folder at `PROJECT_DIR`, or in your Drive, and set `PROJECT_DIR` to it.

In [ ]:
import os, zipfile, glob
if not os.path.exists(os.path.join(PROJECT_DIR, "code", "run_all.py")):
    from google.colab import files
    print("Upload the project archive (.zip):")
    up = files.upload()
    zname = next(iter(up))
    os.makedirs(PROJECT_DIR, exist_ok=True)
    with zipfile.ZipFile(zname) as z:
        z.extractall(PROJECT_DIR)
    # the archive may contain a top-level folder: find the folder that holds code/run_all.py
    hit = glob.glob(os.path.join(PROJECT_DIR, "**", "code", "run_all.py"), recursive=True)
    PROJECT_DIR = os.path.dirname(os.path.dirname(hit[0]))
os.chdir(PROJECT_DIR)
print("project dir:", PROJECT_DIR); print(sorted(os.listdir(".")))

## 2. Install the pinned environment
All steps run as subprocesses, so no runtime restart is needed after the install.

In [ ]:
import subprocess, sys
req = open("requirements.txt").read().splitlines()
if not PIN_TORCH:
    req = [r for r in req if not r.strip().startswith("torch")]
open("/tmp/req_colab.txt", "w").write("\n".join(req) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req_colab.txt"], check=True)
subprocess.run([sys.executable, "-c", "import torch, transformers, datasets, numpy; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, "
                "'| datasets', datasets.__version__, '| numpy', numpy.__version__)"], check=True)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 3. Data
Downloads UD English EWT (HF `universal-dependencies/universal_dependencies`, tag `ud2.18-loader-v2.3.0`), WordSim-353 and the Google analogy set (gensim copies at a pinned commit), and caches `google-bert/bert-base-uncased` @ `86b5e09`. It then prints the sanity report. The other two Project-5 treebanks can be prepared with `P5_TREEBANK=en_gum` or `en_lines`, but only EWT is used here.

In [ ]:
!python code/run_all.py --tasks data

## 4. Task 1 — PPMI + SVD baseline (+ Levy et al. ablations)

In [ ]:
!python code/run_all.py --tasks 1

## 5. Task 2 — structural probes
The steps are feature extraction (all 13 layers, cached to disk), probe training, evaluation, figures and the layer-significance test. The training cell is the long one, and it is resumable.

In [ ]:
flags = ["--tasks", "2"] + (["--fast"] if MODE == "fast" else []) + (["--with-control"] if WITH_CONTROL else [])
print("python code/run_all.py", " ".join(flags))
import subprocess, sys
subprocess.run([sys.executable, "code/run_all.py", *flags], check=True)

## 6. Task 3 — BERT-derived static vectors, comparison with Task 1, extensions, diagnostics

In [ ]:
!python code/run_all.py --tasks 3

## 7. Results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.precision", 3); pd.set_option("display.width", 200)
bl = json.load(open("outputs/task2/best_layer.json"))
display(Markdown(f"**Task 2:** l* = {bl['l_star']} (dev-selected); test UUAS = {100*bl['test']['uuas']:.1f}%, "
                 f"depth Spearman (NSpr) = {bl['test']['nspr']:.3f}"))
display(pd.read_csv("outputs/task2/paper_comparison.csv"))
display(Markdown("**Task 1 vs Task 3 (identical items; T2 probe as reference)**"))
display(pd.read_csv("outputs/task3/comparison_table.csv")[["system", "ws_rho_cov", "ws_rho_full", "an_acc_cov", "an_acc_full", "mst20_uuas", "pool_uuas"]])
display(Markdown("**Diagnostic ladder (same 20 sentences / 660-sentence pool)**"))
display(pd.read_csv("outputs/task3/extensions/ladder.csv")[["system", "mst20_uuas", "pool_uuas"]])

In [ ]:
for f in ["outputs/task2/figures/uuas_vs_layer.png", "outputs/task2/figures/nspr_vs_layer.png",
          "outputs/task3/figures/layer_sweep.png", "outputs/task3/figures/ladder.png",
          "outputs/task2/figures/example_typical.png", "outputs/task3/figures/mst_example.png"]:
    display(Markdown(f"`{f}`")); display(Image(f, width=850))

## 8. (Optional) Save results to Google Drive
This copies `outputs/` without the 5 GB BERT feature cache.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/project5_outputs"
    shutil.copytree("outputs", dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("features"))
    print("saved to", dst)